---

Configure database

---

In [1]:
# autoload
%load_ext autoreload
%autoreload 2

from pgl import pglStorage

In [ ]:
pglStorage.initDatabase()

---

Connect as admin and create the schema

---


In [4]:
from pgl.pglPostgres import pglPostgres

postgres = pglPostgres.fromSettings()
assert postgres is not None, "No PostgreSQL configuration was found."

In [ ]:
postgres.start()

In [3]:
import uuid
from psycopg import sql

class RollbackTest(Exception):
    pass

testUsername = f"pgl_rollback_test_{uuid.uuid4().hex}"
usersTable = sql.Identifier(postgres.databaseSchema, "users")

with postgres.connect() as connection:
    try:
        with connection.transaction():
            statement = sql.SQL("INSERT INTO {} (username, display_name) VALUES (%s, %s) RETURNING user_id").format(usersTable)
            userID = connection.execute(statement, (testUsername, "Temporary rollback test")).fetchone()[0]

            assert userID is not None

            statement = sql.SQL("SELECT username FROM {} WHERE user_id = %s").format(usersTable)
            assert connection.execute(statement, (userID,)).fetchone() == (testUsername,)

            raise RollbackTest()

    except RollbackTest:
        pass

    statement = sql.SQL("SELECT user_id FROM {} WHERE username = %s").format(usersTable)
    assert connection.execute(statement, (testUsername,)).fetchone() is None

print("Application connection and transaction rollback test passed.")

NameError: name 'postgres' is not defined

In [ ]:
testUsername = f"pgl_creator_test_{uuid.uuid4().hex}"
usersTable = sql.Identifier(postgres.databaseSchema, "users")

with postgres.connect() as connection:
    try:
        with connection.transaction():
            firstID = pglStorage.getOrCreateUser(connection, postgres, testUsername, displayName="Original name")
            secondID = pglStorage.getOrCreateUser(connection, postgres, testUsername, displayName="Replacement name")

            assert firstID is not None
            assert firstID == secondID

            statement = sql.SQL("SELECT display_name FROM {} WHERE user_id = %s").format(usersTable)
            assert connection.execute(statement, (firstID,)).fetchone() == ("Original name",)

            raise RollbackTest()

    except RollbackTest:
        pass

    statement = sql.SQL("SELECT user_id FROM {} WHERE username = %s").format(usersTable)
    assert connection.execute(statement, (testUsername,)).fetchone() is None

print("Creator registration, reuse, and rollback tests passed.")

In [ ]:
testBackendName = f"pgl_backend_test_{uuid.uuid4().hex}"
backendRoot = f"memory://{destinationRoot}"
backendsTable = sql.Identifier(postgres.databaseSchema, "storage_backends")

with postgres.connect() as connection:
    try:
        with connection.transaction():
            firstID = pglStorage.getOrCreateStorageBackend(connection, postgres, testBackendName, "local", backendRoot)
            secondID = pglStorage.getOrCreateStorageBackend(connection, postgres, testBackendName, "local", backendRoot)

            assert firstID is not None
            assert firstID == secondID

            try:
                pglStorage.getOrCreateStorageBackend(connection, postgres, testBackendName, "local", "memory:///different-root")
            except ValueError as error:
                assert "different configuration" in str(error)
            else:
                raise AssertionError("Conflicting backend configuration was accepted")

            statement = sql.SQL("SELECT url_prefix FROM {} WHERE backend_id = %s").format(backendsTable)
            assert connection.execute(statement, (firstID,)).fetchone() == (backendRoot,)

            raise RollbackTest()

    except RollbackTest:
        pass

    statement = sql.SQL("SELECT backend_id FROM {} WHERE name = %s").format(backendsTable)
    assert connection.execute(statement, (testBackendName,)).fetchone() is None

print("Storage backend registration, reuse, conflict, and rollback tests passed.")

In [ ]:
import hashlib

testBytes = uuid.uuid4().bytes
testHash = hashlib.sha256(testBytes).hexdigest()
testSize = len(testBytes)
filesTable = sql.Identifier(postgres.databaseSchema, "files")

with postgres.connect() as connection:
    try:
        with connection.transaction():
            firstID = pglStorage.getOrCreateFile(connection, postgres, testHash, testSize)
            secondID = pglStorage.getOrCreateFile(connection, postgres, testHash, testSize)

            assert firstID is not None
            assert firstID == secondID

            try:
                pglStorage.getOrCreateFile(connection, postgres, testHash, testSize + 1)
            except ValueError as error:
                assert "conflicting size or kind" in str(error)
            else:
                raise AssertionError("Conflicting file identity was accepted")

            statement = sql.SQL("SELECT size_bytes, kind FROM {} WHERE file_id = %s").format(filesTable)
            assert connection.execute(statement, (firstID,)).fetchone() == (testSize, "file")

            statement = sql.SQL("SELECT count(*) FROM {} WHERE sha256 = %s").format(filesTable)
            assert connection.execute(statement, (testHash,)).fetchone() == (1,)

            raise RollbackTest()

    except RollbackTest:
        pass

    statement = sql.SQL("SELECT file_id FROM {} WHERE sha256 = %s").format(filesTable)
    assert connection.execute(statement, (testHash,)).fetchone() is None

print("File identity registration, deduplication, conflict, and rollback tests passed.")

In [ ]:
record = staged[0]
testBackendName = f"pgl_location_test_{uuid.uuid4().hex}"
backendRoot = f"memory://{destinationRoot}"
locationsTable = sql.Identifier(postgres.databaseSchema, "storage_locations")

with postgres.connect() as connection:
    try:
        with connection.transaction():
            backendID = pglStorage.getOrCreateStorageBackend(connection, postgres, testBackendName, "local", backendRoot)
            fileID = pglStorage.getOrCreateFile(connection, postgres, record["sha256"], record["size_bytes"])

            firstKey = pglStorage.verifyAndRegisterStorageLocation(connection, postgres, fileID, backendID, record["object_key"])
            secondKey = pglStorage.verifyAndRegisterStorageLocation(connection, postgres, fileID, backendID, record["object_key"])

            assert firstKey == secondKey == record["object_key"]

            statement = sql.SQL("SELECT object_key, status, verified_at FROM {} WHERE file_id = %s AND backend_id = %s").format(locationsTable)
            row = connection.execute(statement, (fileID, backendID)).fetchone()

            assert row[0] == record["object_key"]
            assert row[1] == "present"
            assert row[2] is not None

            try:
                pglStorage.verifyAndRegisterStorageLocation(connection, postgres, fileID, backendID, "different/payload")
            except ValueError as error:
                assert "different object key" in str(error)
            else:
                raise AssertionError("Existing location was redirected")

            raise RollbackTest()

    except RollbackTest:
        pass

    statement = sql.SQL("SELECT 1 FROM {} WHERE backend_id = %s").format(locationsTable)
    assert connection.execute(statement, (backendID,)).fetchone() is None

print("Verified storage-location registration and rollback tests passed.")

In [ ]:
expectedBytes = uuid.uuid4().bytes
wrongBytes = bytes([expectedBytes[0] ^ 1]) + expectedBytes[1:]
expectedHash = hashlib.sha256(expectedBytes).hexdigest()

testKey = f"verification-tests/{uuid.uuid4().hex}/payload"
testPath = posixpath.join(destinationRoot, testKey)
memory.makedirs(posixpath.dirname(testPath), exist_ok=True)
memory.pipe_file(testPath, wrongBytes)

testBackendName = f"pgl_bad_payload_test_{uuid.uuid4().hex}"
backendRoot = f"memory://{destinationRoot}"
locationsTable = sql.Identifier(postgres.databaseSchema, "storage_locations")

try:
    with postgres.connect() as connection:
        try:
            with connection.transaction():
                backendID = pglStorage.getOrCreateStorageBackend(connection, postgres, testBackendName, "local", backendRoot)
                fileID = pglStorage.getOrCreateFile(connection, postgres, expectedHash, len(expectedBytes))

                try:
                    pglStorage.verifyAndRegisterStorageLocation(connection, postgres, fileID, backendID, testKey)
                except ValueError as error:
                    assert "does not match the registered file identity" in str(error)
                else:
                    raise AssertionError("Incorrect payload was accepted")

                statement = sql.SQL("SELECT status FROM {} WHERE file_id = %s AND backend_id = %s").format(locationsTable)
                assert connection.execute(statement, (fileID, backendID)).fetchone() is None

                # Correct the test payload and confirm registration now succeeds.
                memory.pipe_file(testPath, expectedBytes)
                registeredKey = pglStorage.verifyAndRegisterStorageLocation(connection, postgres, fileID, backendID, testKey)

                assert registeredKey == testKey
                assert connection.execute(statement, (fileID, backendID)).fetchone() == ("present",)

                raise RollbackTest()

        except RollbackTest:
            pass

        statement = sql.SQL("SELECT 1 FROM {} WHERE backend_id = %s").format(locationsTable)
        assert connection.execute(statement, (backendID,)).fetchone() is None

finally:
    memory.rm(testPath)

print("Incorrect payload rejected; corrected payload verified; database changes rolled back.")

In [ ]:
testUsername = f"pgl_checkpoint_test_{uuid.uuid4().hex}"
sessionsTable = sql.Identifier(postgres.databaseSchema, "session")

with postgres.connect() as connection:
    try:
        with connection.transaction():
            creatorID = pglStorage.getOrCreateUser(connection, postgres, testUsername)
            checkpointID = pglStorage.createRawCheckpoint(connection, postgres, creatorID, description="Temporary checkpoint test")

            assert checkpointID is not None

            statement = sql.SQL("""
                SELECT root_session_id, parent_session_id, status,
                       manifest_hash, completed_at, created_by, started_at
                FROM {}
                WHERE session_id = %s
            """).format(sessionsTable)

            row = connection.execute(statement, (checkpointID,)).fetchone()

            assert row[0] == checkpointID
            assert row[1] is None
            assert row[2] == "in_progress"
            assert row[3] is None
            assert row[4] is None
            assert row[5] == creatorID
            assert row[6] is not None

            raise RollbackTest()

    except RollbackTest:
        pass

    statement = sql.SQL("SELECT 1 FROM {} WHERE session_id = %s").format(sessionsTable)
    assert connection.execute(statement, (checkpointID,)).fetchone() is None

print("Raw checkpoint identity, lineage, initial state, and rollback tests passed.")

In [ ]:
import psycopg

testUsername = f"pgl_attachment_test_{uuid.uuid4().hex}"
record = staged[0]
linksTable = sql.Identifier(postgres.databaseSchema, "session_files")

with postgres.connect() as connection:
    try:
        with connection.transaction():
            creatorID = pglStorage.getOrCreateUser(connection, postgres, testUsername)
            checkpointID = pglStorage.createRawCheckpoint(connection, postgres, creatorID)
            fileID = pglStorage.getOrCreateFile(connection, postgres, record["sha256"], record["size_bytes"])

            pglStorage.attachCheckpointFile(connection, postgres, checkpointID, record["name"], fileID, role="acquisition")

            statement = sql.SQL("SELECT name, file_id, role, subject_id FROM {} WHERE session_id = %s").format(linksTable)
            expected = (record["name"], fileID, "acquisition", None)
            assert connection.execute(statement, (checkpointID,)).fetchall() == [expected]

            try:
                with connection.transaction():
                    pglStorage.attachCheckpointFile(connection, postgres, checkpointID, record["name"], fileID, role="acquisition")
            except psycopg.errors.UniqueViolation:
                pass
            else:
                raise AssertionError("Duplicate logical filename was accepted")

            assert connection.execute(statement, (checkpointID,)).fetchall() == [expected]

            raise RollbackTest()

    except RollbackTest:
        pass

    statement = sql.SQL("SELECT 1 FROM {} WHERE session_id = %s").format(linksTable)
    assert connection.execute(statement, (checkpointID,)).fetchone() is None

print("Checkpoint file attachment, duplicate rejection, and rollback tests passed.")

In [ ]:
sourceRun = session.runs[0]
settingsPath = posixpath.join(sourceRun.fullDataPath, "experimentSettings.json")

with sourceRun.filesystem.open(settingsPath, "rt", encoding="utf-8") as file:
    recordedSettings = json.load(file)

subjectID = recordedSettings["subjectID"]
assert subjectID == "s0001"

print("Recorded acquisition subject:", subjectID)

In [ ]:
testUsername = f"pgl_subject_test_{uuid.uuid4().hex}"
record = next(record for record in staged if record["name"] == "runs/000000/data.json")
subjectsTable = sql.Identifier(postgres.databaseSchema, "subjects")
sessionSubjectsView = sql.Identifier(postgres.databaseSchema, "session_subjects")

with postgres.connect() as connection:
    statement = sql.SQL("SELECT subject_id, description FROM {} WHERE subject_id = %s").format(subjectsTable)
    originalSubject = connection.execute(statement, (subjectID,)).fetchone()

    try:
        with connection.transaction():
            firstSubject = pglStorage.getOrCreateSubject(connection, postgres, subjectID)
            secondSubject = pglStorage.getOrCreateSubject(connection, postgres, subjectID)
            assert firstSubject == secondSubject == subjectID

            creatorID = pglStorage.getOrCreateUser(connection, postgres, testUsername)
            checkpointID = pglStorage.createRawCheckpoint(connection, postgres, creatorID)
            fileID = pglStorage.getOrCreateFile(connection, postgres, record["sha256"], record["size_bytes"])

            pglStorage.attachCheckpointFile(connection, postgres, checkpointID, record["name"], fileID, role="acquisition", subjectID=subjectID)

            statement = sql.SQL("SELECT subject_id FROM {} WHERE session_id = %s").format(sessionSubjectsView)
            assert connection.execute(statement, (checkpointID,)).fetchall() == [(subjectID,)]

            raise RollbackTest()

    except RollbackTest:
        pass

    statement = sql.SQL("SELECT subject_id, description FROM {} WHERE subject_id = %s").format(subjectsTable)
    assert connection.execute(statement, (subjectID,)).fetchone() == originalSubject

    statement = sql.SQL("SELECT subject_id FROM {} WHERE session_id = %s").format(sessionSubjectsView)
    assert connection.execute(statement, (checkpointID,)).fetchall() == []

print("Subject registration, association, view, and rollback tests passed.")

In [ ]:
testUsername = f"pgl_archive_test_{uuid.uuid4().hex}"
testBackendName = f"pgl_archive_backend_{uuid.uuid4().hex}"
backendRoot = f"memory://{destinationRoot}"
subjectsByRun = {"runs/000000": "s0001"}

sessionsTable = sql.Identifier(postgres.databaseSchema, "session")
linksTable = sql.Identifier(postgres.databaseSchema, "session_files")
filesTable = sql.Identifier(postgres.databaseSchema, "files")
locationsTable = sql.Identifier(postgres.databaseSchema, "storage_locations")

with postgres.connect() as connection:
    try:
        with connection.transaction():
            creatorID = pglStorage.getOrCreateUser(connection, postgres, testUsername)
            backendID = pglStorage.getOrCreateStorageBackend(connection, postgres, testBackendName, "local", backendRoot)

            checkpointID = pglStorage.registerStagedRawCheckpoint(connection, postgres, creatorID, backendID, manifest, staged, storedManifest, subjectsByRun)

            statement = sql.SQL("SELECT status, manifest_hash FROM {} WHERE session_id = %s").format(sessionsTable)
            assert connection.execute(statement, (checkpointID,)).fetchone() == ("in_progress", None)

            statement = sql.SQL("""
                SELECT link.name, content.sha256, content.size_bytes,
                       link.role, link.subject_id, location.status
                FROM {} AS link
                JOIN {} AS content ON content.file_id = link.file_id
                LEFT JOIN {} AS location
                    ON location.file_id = link.file_id AND location.backend_id = %s
                WHERE link.session_id = %s
            """).format(linksTable, filesTable, locationsTable)

            rows = connection.execute(statement, (backendID, checkpointID)).fetchall()
            byName = {row[0]: row[1:] for row in rows}

            assert len(rows) == len(manifest["files"]) + 1

            for record in manifest["files"]:
                assert byName[record["name"]] == (record["sha256"], record["size_bytes"], "acquisition", "s0001", "present")

            assert byName["manifest.json"] == (storedManifest["sha256"], storedManifest["size_bytes"], "manifest", None, "present")

            raise RollbackTest()

    except RollbackTest:
        pass

    statement = sql.SQL("SELECT 1 FROM {} WHERE session_id = %s").format(sessionsTable)
    assert connection.execute(statement, (checkpointID,)).fetchone() is None

    statement = sql.SQL("SELECT 1 FROM {} WHERE session_id = %s").format(linksTable)
    assert connection.execute(statement, (checkpointID,)).fetchone() is None

print("Complete staged archive registration and rollback tests passed.")

In [ ]:
import psycopg

testUsername = f"pgl_complete_test_{uuid.uuid4().hex}"
testBackendName = f"pgl_complete_backend_{uuid.uuid4().hex}"
backendRoot = f"memory://{destinationRoot}"
subjectsByRun = {"runs/000000": "s0001"}

sessionsTable = sql.Identifier(postgres.databaseSchema, "session")

with postgres.connect() as connection:
    try:
        with connection.transaction():
            creatorID = pglStorage.getOrCreateUser(connection, postgres, testUsername)
            backendID = pglStorage.getOrCreateStorageBackend(connection, postgres, testBackendName, "local", backendRoot)

            checkpointID = pglStorage.registerStagedRawCheckpoint(connection, postgres, creatorID, backendID, manifest, staged, storedManifest, subjectsByRun)
            completedHash = pglStorage.completeRawCheckpoint(connection, postgres, checkpointID, manifest)

            assert completedHash == pglStorage.hashRawManifest(manifest)

            statement = sql.SQL("SELECT status, manifest_hash, completed_at FROM {} WHERE session_id = %s").format(sessionsTable)
            row = connection.execute(statement, (checkpointID,)).fetchone()

            assert row[0] == "complete"
            assert row[1] == completedHash
            assert row[2] is not None

            # Use a savepoint so the intentional SQL error does not
            # abort the surrounding test transaction.
            try:
                with connection.transaction():
                    statement = sql.SQL("UPDATE {} SET description = %s WHERE session_id = %s").format(sessionsTable)
                    connection.execute(statement, ("Should be rejected", checkpointID))
            except psycopg.errors.RaiseException as error:
                assert "immutable" in str(error)
            else:
                raise AssertionError("Completed checkpoint was modified")

            raise RollbackTest()

    except RollbackTest:
        pass

    statement = sql.SQL("SELECT 1 FROM {} WHERE session_id = %s").format(sessionsTable)
    assert connection.execute(statement, (checkpointID,)).fetchone() is None

print("Checkpoint completion, immutability, and rollback tests passed.")

In [3]:
import fsspec
import uuid

from pgl.pglSession import pglSession
from pgl.pglSettings import pglSettings
from pgl.pglExperiment import pglExperimentSettings, pglExperimentData, pglTaskSettings, pglTaskState, pglTaskData, pglEventTrial
from pgl.pglStorage import pglStorage
from pgl.pglPostgres import pglPostgres

# Use fresh paths so rerunning this cell doesn't reuse an older fixture.
testRoot = f"/pgl-roundtrip-{uuid.uuid4().hex}"
runPath = f"{testRoot}/source/run01"
destinationRoot = f"{testRoot}/archive"
taskDirectory = "task01_fixation"
taskPath = f"{runPath}/{taskDirectory}"

memory = fsspec.filesystem("memory")
memory.makedirs(taskPath, exist_ok=True)

# Run-level experiment settings.
experimentSettings = pglExperimentSettings()
experimentSettings.subjectID = "s0001"
experimentSettings.sessionName = "session_test"
experimentSettings.runName = "run01"
experimentSettings.tasks = ["fixation"]
experimentSettings.save(filename=f"{runPath}/experimentSettings.json", filesystem=memory)

# Recorded settings: do not discover current displays.
settings = pglSettings(refreshDisplays=False)
settings.responseKeys = "ab"
settings.save(filename=f"{runPath}/settings.json", filesystem=memory)

# Run-level behavioral data.
data = pglExperimentData()
data.startTime = 10.0
data.endTime = 20.0
data.save(filename=f"{runPath}/data.json", filesystem=memory)

# Structural placeholders for this synthetic test only.
memory.pipe_file(f"{runPath}/state.json", b"{}")
memory.pipe_file(f"{runPath}/pgl.json", b"{}")

# Task settings, state, and data.
taskSettings = pglTaskSettings()
taskSettings.taskName = "Fixation"
taskSettings.taskSaveName = "fixation"
taskSettings.taskID = 1
taskSettings.nTrials = 1.0
taskSettings.save(filename=f"{taskPath}/settings.json", filesystem=memory)

taskState = pglTaskState()
taskState.currentTrial = 0
taskState.save(filename=f"{taskPath}/state.json", filesystem=memory)

taskData = pglTaskData()
taskData.startTime = 11.0
taskData.endTime = 12.0
taskData.params = [{"contrast": 0.5}]
taskData.events = [pglEventTrial(trialNum=0, timestamp=11.0)]
taskData.save(filename=f"{taskPath}/data.json", filesystem=memory)

# Preserve the empty directory required by the task loader.
memory.makedirs(f"{taskPath}/parameters", exist_ok=True)

# Build and validate the source manifest.
session = pglSession(filesystem=memory, runList=[runPath])
manifest = pglStorage.buildRawManifest(session)
pglStorage.validateRawManifest(manifest)
manifestHash = pglStorage.hashRawManifest(manifest)

assert len(manifest["files"]) == 8

# Stage and verify acquisition files and the manifest.
staged = pglStorage.stageRawSession(session, manifest, memory, destinationRoot)
storedManifest = pglStorage.stageRawManifest(manifest, memory, destinationRoot)

assert len(staged) == 8
assert storedManifest["sha256"] == manifestHash

# Load the existing database configuration; does not initialize or reset it.
postgres = pglPostgres.fromSettings()
assert postgres is not None, "No PostgreSQL configuration was found."

print("Fresh memory fixture and staged archive ready.")

(pglSettings:save) Saved settings to /pgl-roundtrip-3072114a5c6d4acf846a7ddffb57829f/source/run01/settings.json
Fresh memory fixture and staged archive ready.


In [9]:
import uuid

from psycopg import sql
from unittest.mock import patch
from pgl.pglStorage import pglStorage
from pgl.pglSession import pglSession
from pgl.pglSettings import pglSettings
from pgl import pglCheckpointFilesystem


class RollbackTest(Exception):
    pass


testUsername = f"pgl_roundtrip_test_{uuid.uuid4().hex}"
testBackendName = f"pgl_roundtrip_backend_{uuid.uuid4().hex}"
backendRoot = f"memory://{destinationRoot}"
#subjectsByRun = {"runs/000000": "s0001"}

sessionsTable = sql.Identifier(postgres.databaseSchema, "session")
linksTable = sql.Identifier(postgres.databaseSchema, "session_files")
usersTable = sql.Identifier(postgres.databaseSchema, "users")
backendsTable = sql.Identifier(postgres.databaseSchema, "storage_backends")

pglStorage.validateRawManifest(manifest)

with postgres.connect() as connection:
    try:
        with connection.transaction():
            # Register the creator and test storage backend.
            creatorID = pglStorage.getOrCreateUser(connection, postgres, testUsername)
            backendID = pglStorage.getOrCreateStorageBackend(connection, postgres, testBackendName, "local", backendRoot)

            # Register acquisition files and the manifest.
            checkpointID = pglStorage.registerStagedRawCheckpoint(connection, postgres, creatorID, backendID, manifest, staged, storedManifest)

            statement = sql.SQL("SELECT name, subject_id FROM {} WHERE session_id = %s AND role = 'acquisition'").format(linksTable)
            subjectRows = connection.execute(statement, (checkpointID,)).fetchall()

            assert len(subjectRows) == len(manifest["files"])
            assert {name for name, subjectID in subjectRows} == {record["name"] for record in manifest["files"]}
            assert all(subjectID == "s0001" for name, subjectID in subjectRows)

            statement = sql.SQL("SELECT subject_id FROM {} WHERE session_id = %s AND role = 'manifest'").format(linksTable)
            assert connection.execute(statement, (checkpointID,)).fetchall() == [(None,)]

            print("Subject associations derived from verified archived settings.")

            # Reject loading an unpublished checkpoint.
            try:
                pglStorage.getRawManifestLocations(connection, postgres, checkpointID)
            except ValueError as error:
                assert "Only completed checkpoints" in str(error)
            else:
                raise AssertionError("Unpublished checkpoint was accepted")

            # Complete the checkpoint.
            completedHash = pglStorage.completeRawCheckpoint(connection, postgres, checkpointID, manifest)
            assert completedHash == pglStorage.hashRawManifest(manifest)

            statement = sql.SQL("SELECT status, manifest_hash, completed_at FROM {} WHERE session_id = %s").format(sessionsTable)
            row = connection.execute(statement, (checkpointID,)).fetchone()

            assert row[0] == "complete"
            assert row[1] == completedHash
            assert row[2] is not None

            # Look up manifest identity and storage candidates.
            located = pglStorage.getRawManifestLocations(connection, postgres, checkpointID)

            assert located["sha256"] == completedHash
            assert located["size_bytes"] == len(pglStorage.encodeRawManifest(manifest))

            expectedLocation = {
                "backend_id": backendID,
                "url_prefix": backendRoot,
                "object_key": storedManifest["object_key"],
            }
            assert expectedLocation in located["locations"]

            print("Completed-checkpoint manifest lookup passed.")

            # Fetch, verify, decode, and validate the manifest.
            loadedManifest = pglStorage.loadRawManifest(connection, postgres, checkpointID)

            assert loadedManifest == manifest
            assert pglStorage.hashRawManifest(loadedManifest) == located["sha256"]

            # Check manifest size-limit enforcement.
            try:
                pglStorage.loadRawManifest(connection, postgres, checkpointID, maxBytes=1)
            except ValueError as error:
                assert "exceeds the configured size limit" in str(error)
            else:
                raise AssertionError("Manifest size limit was ignored")

            print("Database-backed manifest retrieval and verification passed.")

            # Retrieve acquisition-file identities and storage candidates.
            fileLocations = pglStorage.getRawFileLocations(connection, postgres, checkpointID, loadedManifest)

            assert set(fileLocations) == {record["name"] for record in loadedManifest["files"]}

            for record in loadedManifest["files"]:
                locatedFile = fileLocations[record["name"]]

                assert locatedFile["sha256"] == record["sha256"]
                assert locatedFile["size_bytes"] == record["size_bytes"]
                assert locatedFile["locations"]

                ourLocations = [location for location in locatedFile["locations"] if location["backend_id"] == backendID]
                assert len(ourLocations) == 1
                assert ourLocations[0]["url_prefix"] == backendRoot

            print("Database acquisition-file location lookup passed.")

            # Build the filesystem from resolved metadata, not a SQL connection.
            databaseCheckpointFS = pglCheckpointFilesystem(loadedManifest, fileLocations)
            archivedRunPaths = [record["path"] for record in loadedManifest["runs"]]

            # Check its logical directory view.
            assert databaseCheckpointFS.isdir("runs/000000")
            assert databaseCheckpointFS.isdir("runs/000000/task01_fixation/parameters")
            assert databaseCheckpointFS.ls("runs/000000/task01_fixation/parameters", detail=False) == []
            assert set(databaseCheckpointFS.find("")) == set(fileLocations)

            loadedSession = pglStorage.loadRawSession(connection, postgres, checkpointID)

            assert len(loadedSession.runs) == 1
            assert loadedSession.runs[0]._experimentSettings is None
            assert loadedSession.runs[0]._settings is None
            assert loadedSession.runs[0]._data is None
            assert loadedSession.runs[0]._tasks is None
            # Never commit the temporary memory-backed checkpoint.
            raise RollbackTest()

    except RollbackTest:
        pass

    # Confirm all test-specific database rows were rolled back.
    statement = sql.SQL("SELECT 1 FROM {} WHERE session_id = %s").format(sessionsTable)
    assert connection.execute(statement, (checkpointID,)).fetchone() is None

    statement = sql.SQL("SELECT 1 FROM {} WHERE session_id = %s").format(linksTable)
    assert connection.execute(statement, (checkpointID,)).fetchone() is None

    statement = sql.SQL("SELECT 1 FROM {} WHERE username = %s").format(usersTable)
    assert connection.execute(statement, (testUsername,)).fetchone() is None

    statement = sql.SQL("SELECT 1 FROM {} WHERE name = %s").format(backendsTable)
    assert connection.execute(statement, (testBackendName,)).fetchone() is None

# The SQL connection is now closed.
assert connection.closed
print("Database changes rolled back and SQL connection closed.")

# Construct and lazily load a session using only the resolved filesystem.
with patch.object(pglSettings, "reloadDisplays", side_effect=AssertionError("Unexpected display refresh")):
    databaseSession = pglSession(filesystem=databaseCheckpointFS, runList=archivedRunPaths)

    assert len(databaseSession.runs) == 1
    archivedRun = databaseSession.runs[0]

    assert archivedRun._experimentSettings is None
    assert archivedRun._settings is None
    assert archivedRun._data is None
    assert archivedRun._tasks is None
    assert archivedRun._taskCache == {}

    assert archivedRun.experimentSettings.subjectID == "s0001"
    assert archivedRun.settings.responseKeys == "ab"
    assert archivedRun.data.startTime == 10.0
    assert archivedRun.data.endTime == 20.0

    task = archivedRun.getTaskAt(0)

    assert task.settings.taskSaveName == "fixation"
    assert task.state.currentTrial == 0
    assert task.data.nTrials == 1
    assert task.data.params == [{"contrast": 0.5}]
    assert task.parameters == []

    # Individual task loading leaves the full list lazy.
    assert archivedRun._tasks is None

    # Both access methods must reuse the same object.
    assert archivedRun.getTaskAt(-1) is task
    assert archivedRun.tasks[0] is task

assert connection.closed

with patch.object(pglSettings, "reloadDisplays", side_effect=AssertionError("Unexpected display refresh")):
    loadedRun = loadedSession.runs[0]

    assert loadedRun.experimentSettings.subjectID == "s0001"
    assert loadedRun.settings.responseKeys == "ab"
    assert loadedRun.data.startTime == 10.0
    assert loadedRun.data.endTime == 20.0

    loadedTask = loadedRun.getTaskAt(0)
    assert loadedTask.settings.taskSaveName == "fixation"
    assert loadedTask.data.nTrials == 1
    assert loadedTask.data.params == [{"contrast": 0.5}]
    assert loadedTask.parameters == []
    assert loadedRun.tasks[0] is loadedTask

print("loadRawSession() passed, including lazy reads after SQL connection closure.")

print("Database-resolved session loads after the SQL connection closes.")
print("Full database-backed archive roundtrip test passed.")

+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
(pglPostgres:_askPassword) Enter Password
+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
Subject associations derived from verified archived settings.
Completed-checkpoint manifest lookup passed.
Database-backed manifest retrieval and verification passed.
Database acquisition-file location lookup passed.
Database changes rolled back and SQL connection closed.
loadRawSession() passed, including lazy reads after SQL connection closure.
Database-resolved session loads after the SQL connection closes.
Full database-backed archive roundtrip test passed.


In [8]:
loadedRun = loadedSession.runs[0]
originalSubjectID = loadedRun.experimentSettings.subjectID

try:
    loadedRun.experimentSettings.subjectID = "s9999"

    recordedSubjects = pglStorage.readRawSubjects(loadedManifest, loadedSession.filesystem)

    assert loadedRun.experimentSettings.subjectID == "s9999"
    assert recordedSubjects == {"runs/000000": "s0001"}

finally:
    loadedRun.experimentSettings.subjectID = originalSubjectID

print("Archived subject associations are independent of in-memory edits.")

Archived subject associations are independent of in-memory edits.


In [5]:
import hashlib
import posixpath
import uuid

from copy import deepcopy
from pgl import pglCheckpointFilesystem

logicalPath = "runs/000000/data.json"

# Use the verified copy belonging to our memory-backed test backend.
goodLocation = next(location for location in fileLocations[logicalPath]["locations"] if location["backend_id"] == backendID)

# Read verified reference bytes.
with databaseCheckpointFS.open(logicalPath, "rb") as file:
    expectedBytes = file.read()

assert expectedBytes
assert hashlib.sha256(expectedBytes).hexdigest() == fileLocations[logicalPath]["sha256"]

# A unique object key that does not exist yet.
badKey = f"fallback-tests/{uuid.uuid4().hex}/payload"
badPath = posixpath.join(destinationRoot, badKey)
badLocation = {**goodLocation, "object_key": badKey}

assert not memory.exists(badPath)

# Put the bad candidate first and the valid candidate second.
fallbackLocations = deepcopy(fileLocations)
fallbackLocations[logicalPath]["locations"] = [badLocation, goodLocation]

fallbackFS = pglCheckpointFilesystem(loadedManifest, fallbackLocations)

# 1. Missing first copy: read should succeed using the second.
with fallbackFS.open(logicalPath, "rb") as file:
    assert file.read() == expectedBytes

print("Missing first replica: fallback passed.")

# 2. Corrupted first copy, with the same size as the valid payload.
corruptedBytes = bytes([expectedBytes[0] ^ 1]) + expectedBytes[1:]
memory.makedirs(posixpath.dirname(badPath), exist_ok=True)

try:
    memory.pipe_file(badPath, corruptedBytes)

    with fallbackFS.open(logicalPath, "rb") as file:
        assert file.read() == expectedBytes

    print("Corrupted first replica: fallback passed.")

    # 3. Only the corrupted copy remains: reading must fail.
    brokenLocations = deepcopy(fileLocations)
    brokenLocations[logicalPath]["locations"] = [badLocation]
    brokenFS = pglCheckpointFilesystem(loadedManifest, brokenLocations)

    try:
        with brokenFS.open(logicalPath, "rb") as file:
            file.read()
    except OSError as error:
        assert "No verified copy available" in str(error)
        print("All replicas invalid: correctly rejected.")
    else:
        raise AssertionError("Corrupted payload was returned")

finally:
    if memory.exists(badPath):
        memory.rm(badPath)

# Confirm the original archive remains readable.
with databaseCheckpointFS.open(logicalPath, "rb") as file:
    assert file.read() == expectedBytes

print("Replica fallback tests passed; original archive unchanged.")

Missing first replica: fallback passed.
Corrupted first replica: fallback passed.
All replicas invalid: correctly rejected.
Replica fallback tests passed; original archive unchanged.


In [10]:
import uuid
from pathlib import Path
from fsspec.core import url_to_fs

from pgl.pglSession import pglSession
from pgl.pglStorage import pglStorage

# A fresh, dedicated persistent archive directory.
persistentDirectory = Path.home() / "data" / "pgl-checkpoint-tests" / uuid.uuid4().hex
persistentBackendRoot = persistentDirectory.as_uri()

# Resolve the destination through fsspec.
persistentFS, persistentRoot = url_to_fs(persistentBackendRoot)

print("Persistent archive directory:", persistentDirectory)
print("Backend URL:", persistentBackendRoot)

# Reconstruct the source session from the synthetic acquisition files.
sourceSession = pglSession(filesystem=memory, runList=[runPath])

persistentManifest = pglStorage.buildRawManifest(sourceSession)
pglStorage.validateRawManifest(persistentManifest)

persistentStaged = pglStorage.stageRawSession(sourceSession, persistentManifest, persistentFS, persistentRoot)
persistentStoredManifest = pglStorage.stageRawManifest(persistentManifest, persistentFS, persistentRoot)

assert len(persistentStaged) == len(persistentManifest["files"]) == 8
assert persistentStoredManifest["sha256"] == pglStorage.hashRawManifest(persistentManifest)

print("Persistent archive staged successfully.")

Persistent archive directory: /Users/justin/data/pgl-checkpoint-tests/3f9db33d7e97493bae6f7bf248164e70
Backend URL: file:///Users/justin/data/pgl-checkpoint-tests/3f9db33d7e97493bae6f7bf248164e70
Persistent archive staged successfully.


In [11]:
import json
import posixpath

verificationFS, verificationRoot = url_to_fs(persistentBackendRoot)

for record in persistentStaged:
    payloadPath = posixpath.join(verificationRoot, record["object_key"])
    identity = pglStorage.hashFile(verificationFS, payloadPath)

    assert identity["sha256"] == record["sha256"]
    assert identity["size_bytes"] == record["size_bytes"]

manifestPath = posixpath.join(verificationRoot, persistentStoredManifest["object_key"])

with verificationFS.open(manifestPath, "rb") as file:
    manifestBytes = file.read()

assert manifestBytes == pglStorage.encodeRawManifest(persistentManifest)
assert json.loads(manifestBytes) == persistentManifest

print("All persistent payloads and the manifest verified.")

All persistent payloads and the manifest verified.


In [12]:
import uuid
from psycopg import sql

persistentManifestHash = pglStorage.hashRawManifest(persistentManifest)
persistentBackendName = f"local_roundtrip_test_{uuid.uuid4().hex}"
creatorName = "pgl_roundtrip_test"

sessionsTable = sql.Identifier(postgres.databaseSchema, "session")

with postgres.connect() as connection:
    with connection.transaction():
        # Serialize cooperating ingestions of this manifest.
        connection.execute("SELECT pg_advisory_xact_lock(hashtextextended(%s, 0))", (persistentManifestHash,))

        statement = sql.SQL("SELECT session_id, status, contract_version FROM {} WHERE manifest_hash = %s AND origin = 'raw'").format(sessionsTable)
        existing = connection.execute(statement, (persistentManifestHash,)).fetchone()

        if existing is not None and existing[1:] != ("complete", "pgl.raw.behavior.v1"):
            raise ValueError("Matching manifest belongs to an incomplete or unsupported checkpoint; inspect it before proceeding.")

        persistentBackendID = pglStorage.getOrCreateStorageBackend(connection, postgres, persistentBackendName, "local", persistentBackendRoot)

        if existing is None:
            creatorID = pglStorage.getOrCreateUser(connection, postgres, creatorName, displayName="PGL roundtrip test")

            persistentSessionID = pglStorage.registerStagedRawCheckpoint(connection, postgres, creatorID, persistentBackendID, persistentManifest, persistentStaged, persistentStoredManifest)
            completedHash = pglStorage.completeRawCheckpoint(connection, postgres, persistentSessionID, persistentManifest)

            assert completedHash == persistentManifestHash
            checkpointAction = "Created"

        else:
            persistentSessionID = existing[0]

            # Register this persistent replica without changing the completed
            # checkpoint or its file links. Reuse one object per content hash.
            registeredHashes = set()

            for record in [*persistentStaged, persistentStoredManifest]:
                if record["sha256"] in registeredHashes:
                    continue

                fileID = pglStorage.getOrCreateFile(connection, postgres, record["sha256"], record["size_bytes"])
                pglStorage.verifyAndRegisterStorageLocation(connection, postgres, fileID, persistentBackendID, record["object_key"])
                registeredHashes.add(record["sha256"])

            checkpointAction = "Reused"

        # Check consistency and confirm every file has this persistent backend.
        manifestLocations = pglStorage.getRawManifestLocations(connection, postgres, persistentSessionID)
        assert any(location["backend_id"] == persistentBackendID for location in manifestLocations["locations"])

        persistentLocations = pglStorage.getRawFileLocations(connection, postgres, persistentSessionID, persistentManifest)

        for record in persistentLocations.values():
            assert any(location["backend_id"] == persistentBackendID for location in record["locations"])

    # Normal transaction exit commits the checkpoint/replica registration.

print(f"{checkpointAction} persistent checkpoint: {persistentSessionID}")
print("Manifest SHA-256:", persistentManifestHash)
print("Persistent backend:", persistentBackendRoot)

+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
(pglPostgres:_askPassword) Enter Password
+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
Created persistent checkpoint: 15
Manifest SHA-256: a159d915dbdcca1ca6d9f14e6d63ec41b5d9056787ec4c39217801c8075bcaa0
Persistent backend: file:///Users/justin/data/pgl-checkpoint-tests/3f9db33d7e97493bae6f7bf248164e70


In [13]:
import json
import posixpath

receipt = {
    "session_id": persistentSessionID,
    "manifest_hash": persistentManifestHash,
    "backend_id": persistentBackendID,
    "backend_root": persistentBackendRoot,
    "database_name": postgres.databaseName,
    "database_schema": postgres.databaseSchema,
}

receiptPath = posixpath.join(persistentRoot, "roundtrip-receipt.json")

with persistentFS.open(receiptPath, "wt", encoding="utf-8") as file:
    json.dump(receipt, file, indent=2)

print("SAVE THIS RECEIPT URL:")
print(persistentBackendRoot.rstrip("/") + "/roundtrip-receipt.json")

SAVE THIS RECEIPT URL:
file:///Users/justin/data/pgl-checkpoint-tests/3f9db33d7e97493bae6f7bf248164e70/roundtrip-receipt.json


In [1]:
import json
import fsspec

from unittest.mock import patch
from pgl.pglPostgres import pglPostgres
from pgl.pglStorage import pglStorage
from pgl.pglSettings import pglSettings

receiptURL = "file:///Users/justin/data/pgl-checkpoint-tests/3f9db33d7e97493bae6f7bf248164e70/roundtrip-receipt.json"

with fsspec.open(receiptURL, "rt", encoding="utf-8") as file:
    receipt = json.load(file)

postgres = pglPostgres.fromSettings()
assert postgres is not None, "No PostgreSQL configuration was found."
assert postgres.databaseName == receipt["database_name"]
assert postgres.databaseSchema == receipt["database_schema"]

with patch.object(pglSettings, "reloadDisplays", side_effect=AssertionError("Unexpected display refresh")):
    with postgres.connect() as connection:
        identity = pglStorage.getRawManifestLocations(connection, postgres, receipt["session_id"])
        assert identity["sha256"] == receipt["manifest_hash"]

        session = pglStorage.loadRawSession(connection, postgres, receipt["session_id"])

        assert len(session.runs) == 1
        run = session.runs[0]

        # Acquisition data should still be lazy.
        assert run._experimentSettings is None
        assert run._settings is None
        assert run._data is None
        assert run._tasks is None

    # All following reads happen after the SQL connection closes.
    assert connection.closed

    assert run.experimentSettings.subjectID == "s0001"
    assert run.experimentSettings.sessionName == "session_test"
    assert run.settings.responseKeys == "ab"
    assert run.data.startTime == 10.0
    assert run.data.endTime == 20.0

    task = run.getTaskAt(0)
    assert task.settings.taskSaveName == "fixation"
    assert task.state.currentTrial == 0
    assert task.data.nTrials == 1
    assert task.data.params == [{"contrast": 0.5}]
    assert task.parameters == []

    assert run._tasks is None
    assert run.tasks[0] is task
    assert run.getTaskAt(-1) is task

    assert session.filesystem.isdir("runs/000000/task01_fixation/parameters")
    assert session.filesystem.ls("runs/000000/task01_fixation/parameters", detail=False) == []

print("Fresh-kernel persistent checkpoint roundtrip passed.")
print("Loaded session ID:", receipt["session_id"])
print("Lazy reads succeeded after the SQL connection closed.")

+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
(pglPostgres:_askPassword) Enter Password
+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
Fresh-kernel persistent checkpoint roundtrip passed.
Loaded session ID: 15
Lazy reads succeeded after the SQL connection closed.


In [6]:
from psycopg import sql
from unittest.mock import patch
from pgl.pglStorage import pglStorage

sessionID = 15
sessionsTable = sql.Identifier(postgres.databaseSchema, "session")

with postgres.connect() as connection:
    sourceSession = pglStorage.loadRawSession(connection, postgres, sessionID)

    statement = sql.SQL("SELECT created_by, manifest_hash FROM {} WHERE session_id = %s").format(sessionsTable)
    row = connection.execute(statement, (sessionID,)).fetchone()
    assert row is not None

    creatorID, manifestHash = row

    # Select an active, verified backend holding the manifest.
    manifestLocations = pglStorage.getRawManifestLocations(connection, postgres, sessionID)
    backendID = manifestLocations["locations"][0]["backend_id"]

    with patch.object(pglStorage, "stageRawSession", side_effect=AssertionError("Unexpected acquisition upload")):
        with patch.object(pglStorage, "stageRawManifest", side_effect=AssertionError("Unexpected manifest upload")):
            savedID = pglStorage.saveRawSession(connection, postgres, sourceSession, creatorID, backendID)

    assert savedID == sessionID

    statement = sql.SQL("SELECT count(*) FROM {} WHERE manifest_hash = %s AND origin = 'raw'").format(sessionsTable)
    assert connection.execute(statement, (manifestHash,)).fetchone() == (1,)

print("saveRawSession() reused checkpoint", savedID, "without uploading payloads.")

+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
(pglPostgres:_askPassword) Enter Password
+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
saveRawSession() reused checkpoint 15 without uploading payloads.


In [7]:
import fsspec
import posixpath
import tempfile
import uuid

from pathlib import Path
from psycopg import sql
from fsspec.core import url_to_fs
from pgl.pglStorage import pglStorage
from pgl.pglSession import pglSession


class RollbackNewCheckpointTest(Exception):
    pass


memory = fsspec.filesystem("memory")
fixtureRoot = f"/pgl-new-checkpoint-test/{uuid.uuid4().hex}"
sessionsTable = sql.Identifier(postgres.databaseSchema, "session")

with tempfile.TemporaryDirectory(prefix="pgl-checkpoint-test-") as temporaryDirectory:
    destinationURL = Path(temporaryDirectory).resolve().as_uri()
    destinationFS, destinationPath = url_to_fs(destinationURL)

    with postgres.connect() as connection:
        # Read the existing archive; never modify its payloads.
        source = pglStorage.loadRawSession(connection, postgres, 15)
        sourceManifest = pglStorage.loadRawManifest(connection, postgres, 15)

        # Copy its logical directory tree into an independent memory fixture.
        for directory in sourceManifest["directories"]:
            memory.makedirs(posixpath.join(fixtureRoot, directory), exist_ok=True)

        for record in sourceManifest["files"]:
            with source.filesystem.open(record["name"], "rb") as file:
                memory.pipe_file(posixpath.join(fixtureRoot, record["name"]), file.read())

        fixtureRunPaths = [posixpath.join(fixtureRoot, record["path"]) for record in sourceManifest["runs"]]

        # Ensure this fixture has a new content identity.
        markerPath = posixpath.join(fixtureRunPaths[0], "roundtrip-test-id.txt")
        memory.pipe_file(markerPath, uuid.uuid4().hex.encode("ascii"))

        fixtureSession = pglSession(filesystem=memory, runList=fixtureRunPaths)
        fixtureManifest = pglStorage.buildRawManifest(fixtureSession)
        fixtureHash = pglStorage.hashRawManifest(fixtureManifest)

        statement = sql.SQL("SELECT 1 FROM {} WHERE manifest_hash = %s").format(sessionsTable)
        assert connection.execute(statement, (fixtureHash,)).fetchone() is None

        try:
            with connection.transaction():
                creatorID = pglStorage.getOrCreateUser(connection, postgres, f"pgl_new_test_{uuid.uuid4().hex}")
                backendID = pglStorage.getOrCreateStorageBackend(connection, postgres, f"pgl_new_backend_{uuid.uuid4().hex}", "local", destinationURL)

                # The helper's transaction is nested inside our rollback test.
                newID = pglStorage.saveRawSession(connection, postgres, fixtureSession, creatorID, backendID)

                assert newID != 15

                statement = sql.SQL("SELECT status, manifest_hash FROM {} WHERE session_id = %s").format(sessionsTable)
                assert connection.execute(statement, (newID,)).fetchone() == ("complete", fixtureHash)

                restoredManifest = pglStorage.loadRawManifest(connection, postgres, newID)
                assert restoredManifest == fixtureManifest

                restoredSession = pglStorage.loadRawSession(connection, postgres, newID)
                restoredRun = restoredSession.runs[0]

                assert restoredRun._data is None
                assert restoredRun.data.startTime == 10.0
                assert restoredRun.getTaskAt(0).data.params == [{"contrast": 0.5}]

                # Compare every archived file, including the unique marker.
                for record in fixtureManifest["files"]:
                    with restoredSession.filesystem.open(record["name"], "rb") as file:
                        restoredBytes = file.read()

                    with memory.open(posixpath.join(fixtureRoot, record["name"]), "rb") as file:
                        assert restoredBytes == file.read()

                assert destinationFS.find(destinationPath)
                print("New checkpoint saved, completed, and loaded successfully.")

                raise RollbackNewCheckpointTest()

        except RollbackNewCheckpointTest:
            pass

        statement = sql.SQL("SELECT 1 FROM {} WHERE session_id = %s").format(sessionsTable)
        assert connection.execute(statement, (newID,)).fetchone() is None

        print("Test database changes rolled back.")

    # TemporaryDirectory removes payloads only after database rollback
    # and connection closure.

memory.rm(fixtureRoot, recursive=True)

print("New-checkpoint test passed; temporary source and archive removed.")

+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
(pglPostgres:_askPassword) Enter Password
+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
New checkpoint saved, completed, and loaded successfully.
Test database changes rolled back.
New-checkpoint test passed; temporary source and archive removed.


In [8]:
import fsspec
import posixpath
import tempfile
import uuid

from pathlib import Path
from unittest.mock import patch
from psycopg import sql
from pgl.pglStorage import pglStorage
from pgl.pglSession import pglSession


class InjectedSaveFailure(Exception):
    pass


class RollbackFailureTest(Exception):
    pass


memory = fsspec.filesystem("memory")
fixtureRoot = f"/pgl-failure-test/{uuid.uuid4().hex}"

sessionsTable = sql.Identifier(postgres.databaseSchema, "session")
linksTable = sql.Identifier(postgres.databaseSchema, "session_files")
locationsTable = sql.Identifier(postgres.databaseSchema, "storage_locations")
filesTable = sql.Identifier(postgres.databaseSchema, "files")

# Capture the real method before patching it.
realComplete = pglStorage.completeRawCheckpoint
attemptedIDs = []


def completeThenFail(connection, postgres, sessionID, manifest):
    attemptedIDs.append(sessionID)
    realComplete(connection, postgres, sessionID, manifest)
    raise InjectedSaveFailure("Intentional failure after completion update")


try:
    with tempfile.TemporaryDirectory(prefix="pgl-failure-test-") as temporaryDirectory:
        destinationURL = Path(temporaryDirectory).resolve().as_uri()

        with postgres.connect() as connection:
            source = pglStorage.loadRawSession(connection, postgres, 15)
            sourceManifest = pglStorage.loadRawManifest(connection, postgres, 15)

            # Create an independent acquisition fixture.
            for directory in sourceManifest["directories"]:
                memory.makedirs(posixpath.join(fixtureRoot, directory), exist_ok=True)

            for record in sourceManifest["files"]:
                with source.filesystem.open(record["name"], "rb") as file:
                    memory.pipe_file(posixpath.join(fixtureRoot, record["name"]), file.read())

            fixtureRunPaths = [posixpath.join(fixtureRoot, record["path"]) for record in sourceManifest["runs"]]
            markerPath = posixpath.join(fixtureRunPaths[0], "failure-test-id.txt")
            memory.pipe_file(markerPath, uuid.uuid4().hex.encode("ascii"))

            fixtureSession = pglSession(filesystem=memory, runList=fixtureRunPaths)
            fixtureManifest = pglStorage.buildRawManifest(fixtureSession)
            fixtureHash = pglStorage.hashRawManifest(fixtureManifest)

            # The unique marker must not already have a registered identity.
            markerRecord = next(record for record in fixtureManifest["files"] if record["name"].endswith("/failure-test-id.txt"))
            statement = sql.SQL("SELECT 1 FROM {} WHERE sha256 = %s").format(filesTable)
            assert connection.execute(statement, (markerRecord["sha256"],)).fetchone() is None

            try:
                with connection.transaction():
                    creatorID = pglStorage.getOrCreateUser(connection, postgres, f"pgl_failure_user_{uuid.uuid4().hex}")
                    backendID = pglStorage.getOrCreateStorageBackend(connection, postgres, f"pgl_failure_backend_{uuid.uuid4().hex}", "local", destinationURL)

                    # Catch outside saveRawSession(), allowing its internal
                    # transaction/savepoint to roll back first.
                    try:
                        with patch.object(pglStorage, "completeRawCheckpoint", side_effect=completeThenFail):
                            pglStorage.saveRawSession(connection, postgres, fixtureSession, creatorID, backendID)
                    except InjectedSaveFailure:
                        pass
                    else:
                        raise AssertionError("Injected failure did not propagate")

                    assert len(attemptedIDs) == 1
                    failedID = attemptedIDs[0]

                    statement = sql.SQL("SELECT 1 FROM {} WHERE session_id = %s").format(sessionsTable)
                    assert connection.execute(statement, (failedID,)).fetchone() is None

                    statement = sql.SQL("SELECT 1 FROM {} WHERE session_id = %s").format(linksTable)
                    assert connection.execute(statement, (failedID,)).fetchone() is None

                    statement = sql.SQL("SELECT 1 FROM {} WHERE backend_id = %s").format(locationsTable)
                    assert connection.execute(statement, (backendID,)).fetchone() is None

                    statement = sql.SQL("SELECT 1 FROM {} WHERE sha256 = %s").format(filesTable)
                    assert connection.execute(statement, (markerRecord["sha256"],)).fetchone() is None

                    statement = sql.SQL("SELECT 1 FROM {} WHERE manifest_hash = %s").format(sessionsTable)
                    assert connection.execute(statement, (fixtureHash,)).fetchone() is None

                    print("Failed save rolled back checkpoint, links, locations, and new file identity.")

                    # Payload writes are outside the SQL transaction.
                    assert any(path.is_file() for path in Path(temporaryDirectory).rglob("*"))
                    print("Unregistered staging payloads remain, as expected.")

                    # Remove the test creator/backend via outer rollback.
                    raise RollbackFailureTest()

            except RollbackFailureTest:
                pass

        # TemporaryDirectory now removes the unregistered payloads.

finally:
    if memory.exists(fixtureRoot):
        memory.rm(fixtureRoot, recursive=True)

print("Injected-failure rollback test passed; temporary payloads cleaned up.")

+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
(pglPostgres:_askPassword) Enter Password
+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
Failed save rolled back checkpoint, links, locations, and new file identity.
Unregistered staging payloads remain, as expected.
Injected-failure rollback test passed; temporary payloads cleaned up.
